#  Week 6, Day 1 — June 22, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")
from sklearn.model_selection import train_test_split

## Step 1: Load Datasets

In [ ]:
# Primary dataset for RF modeling:
# realistic_ocean_climate_dataset — 500 rows, SST + pH + Species in same row
df_c = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
df_c['year']  = df_c['Date'].dt.year
df_c['month'] = df_c['Date'].dt.month

# Master Table — for reference and plastic density values
master = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')

print(f'climate dataset : {df_c.shape}')
print(f'master table    : {master.shape}')
print()
print('Climate dataset columns:')
print(df_c.columns.tolist())

## Step 2: Define Independent & Dependent Variables

In [ ]:

FEATURES_RF = ['SST (°C)', 'pH Level']

TARGET_RF = 'Species Observed'

print('VARIABLE DEFINITION')
print('='*55)
print()
print('INDEPENDENT VARIABLES (X):')
print('  [1] SST (°C)   — Sea Surface Temperature')
print('       Justification: Pearson r=-0.6813, p=1.79e-69')
print('       Tipping point: 30.6286°C (2.13°C above current mean)')
print()
print('  [2] pH Level   — Ocean acidity')
print('       Justification: Pearson r=+0.3270, p=6.38e-14')
print('       Tipping point: 7.8795 (~85 years at 0.002 pH/yr decline)')
print()
print('DEPENDENT VARIABLE (y):')
print('  [3] Species Observed — Marine species population count')
print('       Range  : 54 – 171 species')
print('       Mean   : 120.47 species')
print('       Std dev: 20.48 species')

## Step 3: Extract Modeling Dataset

In [ ]:
rf_data = df_c[FEATURES_RF + [TARGET_RF]].dropna().copy()

print(f'Modeling dataset: {rf_data.shape[0]} rows × {len(FEATURES_RF)} features + 1 target')
print()
print('Feature statistics:')
print(rf_data[FEATURES_RF].describe().round(4))
print()
print('Target statistics:')
print(rf_data[[TARGET_RF]].describe().round(4))
print()
print('Why this dataset?')
print('  realistic_ocean_climate_dataset: SST, pH, and Species in the SAME row')
print('   500 records — sufficient for RF training + test split')
print('   Master Table has only 10 joint rows (plastic + species both present)')
print('   — insufficient for supervised learning')

## Step 4: Apply Train / Test Split (80% / 20%)

In [ ]:
X = rf_data[FEATURES_RF]
y = rf_data[TARGET_RF]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42   # fixed seed for reproducibility
)

print('TRAIN / TEST SPLIT (80% / 20%)')
print('='*50)
print(f'  Total records  : {len(rf_data)}')
print(f'  Training set   : {len(X_train)} records  (80%)')
print(f'  Test set       : {len(X_test)}  records  (20%)')
print(f'  random_state   : 42  (reproducibility)')
print()
print('Training feature statistics:')
print(X_train.describe().round(4))
print()
print('Test target statistics:')
print(y_test.describe().round(4))

## Step 5: Verify No Target Leakage

In [ ]:
# Confirm test set indices do not overlap with training set
train_idx = set(X_train.index)
test_idx  = set(X_test.index)
overlap   = train_idx.intersection(test_idx)

print('DATA LEAKAGE CHECK')
print('='*40)
print(f'  Training indices   : {len(train_idx)} unique')
print(f'  Test indices       : {len(test_idx)} unique')
print(f'  Overlapping rows   : {len(overlap)}')
print()
if len(overlap) == 0:
    print('  No data leakage — training and test sets are completely separate')
else:
    print('  Leakage detected — check split parameters')

print()
print('Target distribution comparison (train vs test):')
print(f'  Train y — mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min():.0f}, max={y_train.max():.0f}')
print(f'  Test  y — mean={y_test.mean():.2f}, std={y_test.std():.2f}, '
      f'min={y_test.min():.0f}, max={y_test.max():.0f}')
print()
print('  Train and test distributions are comparable — split is valid')

## Step 6: Visualize Feature–Target Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.set_style('whitegrid')

# ── Panel 1: SST vs Species Observed (colored by pH) ──
ax1 = axes[0]
sc1 = ax1.scatter(X_train['SST (°C)'], y_train,
                  c=X_train['pH Level'], cmap='RdYlBu',
                  s=20, alpha=0.6, edgecolors='none')
plt.colorbar(sc1, ax=ax1, label='pH Level')
# Add regression line (from Pearson analysis: r=-0.6813)
z1 = np.polyfit(X_train['SST (°C)'], y_train, 1)
x_range = np.linspace(X_train['SST (°C)'].min(), X_train['SST (°C)'].max(), 100)
ax1.plot(x_range, np.poly1d(z1)(x_range), 'r--', linewidth=2,
         label=f'Trend (r=-0.6813)')
ax1.set_xlabel('SST (°C)', fontsize=11)
ax1.set_ylabel('Species Observed', fontsize=11)
ax1.set_title('SST vs Species
(colored by pH)', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# ── Panel 2: pH vs Species Observed (colored by SST) ──
ax2 = axes[1]
sc2 = ax2.scatter(X_train['pH Level'], y_train,
                  c=X_train['SST (°C)'], cmap='RdYlBu_r',
                  s=20, alpha=0.6, edgecolors='none')
plt.colorbar(sc2, ax=ax2, label='SST (°C)')
z2 = np.polyfit(X_train['pH Level'], y_train, 1)
x_range2 = np.linspace(X_train['pH Level'].min(), X_train['pH Level'].max(), 100)
ax2.plot(x_range2, np.poly1d(z2)(x_range2), 'b--', linewidth=2,
         label=f'Trend (r=+0.3270)')
ax2.set_xlabel('pH Level', fontsize=11)
ax2.set_ylabel('Species Observed', fontsize=11)
ax2.set_title('pH vs Species
(colored by SST)', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# ── Panel 3: Target distribution ──
ax3 = axes[2]
ax3.hist(y_train, bins=20, color='steelblue', alpha=0.7,
         edgecolor='white', label=f'Train (n={len(y_train)})')
ax3.hist(y_test, bins=20, color='#e74c3c', alpha=0.5,
         edgecolor='white', label=f'Test (n={len(y_test)})')
ax3.axvline(y_train.mean(), color='steelblue', linestyle='--',
            linewidth=2, label=f'Train mean={y_train.mean():.1f}')
ax3.axvline(y_test.mean(), color='#e74c3c', linestyle='--',
            linewidth=2, label=f'Test mean={y_test.mean():.1f}')
ax3.set_xlabel('Species Observed', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title('Target Distribution
Train vs Test', fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

plt.suptitle('June 22, 2026 — Feature–Target Relationships
'
             'Random Forest Input: SST & pH → Species Observed',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week6_Day1_rf_feature_target.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Feature-target chart saved ')

## Step 7: Save Split Metadata

In [ ]:
with open(DIRS['metadata']+'/rf_split_meta.json', 'w') as f:
    json.dump({
        'model_type':   'RandomForestRegressor',
        'features':     FEATURES_RF,
        'target':       TARGET_RF,
        'n_total':      int(len(rf_data)),
        'n_train':      int(len(X_train)),
        'n_test':       int(len(X_test)),
        'test_size':    0.2,
        'random_state': 42,
        'target_stats': {
            'min':  int(y.min()),
            'max':  int(y.max()),
            'mean': round(float(y.mean()), 2),
            'std':  round(float(y.std()), 2),
        },
        'train_feature_means': {
            'SST_C':     round(float(X_train['SST (°C)'].mean()), 4),
            'pH_Level':  round(float(X_train['pH Level'].mean()), 4),
        },
        'generated': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    }, f, indent=2)

print('rf_split_meta.json saved ')
print()
print('DATASET SETUP COMPLETE — SUMMARY')
print('='*50)
print(f'  Independent variables (X)  : {FEATURES_RF}')
print(f'  Dependent variable   (y)   : {TARGET_RF}')
print(f'  Training set               : {len(X_train)} records (80%)')
print(f'  Test set                   : {len(X_test)}  records (20%)')
print(f'  Target range               : {int(y.min())} – {int(y.max())} species')
print(f'  Target mean                : {y.mean():.2f} species')
print(f'  Data leakage               : None ✅')
print()
print('Ready for Random Forest training in June 23 notebook.')